# Arhitektura modela — LSTM baseline vs. BERT fine-tuning

U ovom notebooku upoređujemo dve arhitekture za klasifikaciju support tiketa u **10 queue kategorija**:

1. **LSTMTicketClassifier** — baseline sa embedding slojem, bidirectional LSTM i fully connected head-om
2. **BertTicketClassifier** — fine-tuning pretrained `bert-base-uncased` modela

Oba modela su kompatibilna sa postojećim `TicketDataset` iz [`src/preprocessing.py`](../src/preprocessing.py).

## 1. Priprema okruženja

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import seaborn as sns
import torch

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.model import BertTicketClassifier, LSTMTicketClassifier
from src.preprocessing import TextPreprocessor

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Inicijalizacija modela

In [ ]:
lstm_model = LSTMTicketClassifier(num_classes=config.NUM_CLASSES)
bert_model = BertTicketClassifier(num_classes=config.NUM_CLASSES)

print(f"LSTM embedding init: {lstm_model.embedding_init}")
print(f"GloVe putanja (opciono): {config.GLOVE_PATH}")
print(f"GloVe postoji: {config.GLOVE_PATH.exists()}")

## 3. Opis arhitekture

In [ ]:
print(lstm_model.describe_architecture())
print()
print(bert_model.describe_architecture())

## 4. Broj parametara

In [ ]:
param_rows = [
    {"model": "LSTM", "type": "total", "count": lstm_model.count_parameters()["total"]},
    {"model": "LSTM", "type": "trainable", "count": lstm_model.count_parameters()["trainable"]},
    {"model": "BERT", "type": "total", "count": bert_model.count_parameters()["total"]},
    {"model": "BERT", "type": "trainable", "count": bert_model.count_parameters()["trainable"]},
]
param_df = pd.DataFrame(param_rows)
param_df

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=param_df, x="model", y="count", hue="type", palette="Set2")
plt.title("Ukupan broj parametara po modelu")
plt.ylabel("Broj parametara")
plt.xlabel("Model")
plt.ticklabel_format(style="plain", axis="y")
plt.tight_layout()
plt.show()

## 5. Poređenje parametara po slojevima

In [ ]:
lstm_groups = lstm_model.parameter_groups()
bert_groups = bert_model.parameter_groups()

layer_rows = []
for layer, count in lstm_groups.items():
    layer_rows.append({"model": "LSTM", "layer": layer, "count": count})
for layer, count in bert_groups.items():
    layer_rows.append({"model": "BERT", "layer": layer, "count": count})

layer_df = pd.DataFrame(layer_rows)
layer_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model_name in zip(axes, ["LSTM", "BERT"]):
    subset = layer_df[layer_df["model"] == model_name]
    sns.barplot(data=subset, x="layer", y="count", hue="layer", ax=ax, palette="crest", legend=False)
    ax.set_title(f"{model_name} — parametri po sloju")
    ax.set_xlabel("Sloj")
    ax.set_ylabel("Broj parametara")
    ax.ticklabel_format(style="plain", axis="y")

plt.tight_layout()
plt.show()

## 6. Dummy forward pass

Testiramo oba modela na malom batch-u iz `train.csv`.

In [ ]:
preprocessor = TextPreprocessor()
train_df = pd.read_csv(config.TRAIN_FILE)
dataset = preprocessor.create_dataset(train_df.head(32))

batch = {
    key: torch.stack([dataset[i][key] for i in range(len(dataset))])
    for key in dataset[0].keys()
}

lstm_out = lstm_model(batch["input_ids"], batch["attention_mask"], batch["labels"])
bert_out = bert_model(batch["input_ids"], batch["attention_mask"], batch["labels"])

print(f"Batch shape: {batch['input_ids'].shape}")
print(f"LSTM logits: {lstm_out['logits'].shape}, probs: {lstm_out['probs'].shape}, loss: {lstm_out['loss'].item():.4f}")
print(f"BERT logits: {bert_out['logits'].shape}, probs: {bert_out['probs'].shape}, loss: {bert_out['loss'].item():.4f}")

## 7. Dijagram arhitekture

In [ ]:
def draw_architecture_diagram(layers, title, ax):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, len(layers) * 2 + 1)
    ax.axis("off")
    ax.set_title(title, fontsize=13, pad=12)

    for idx, (name, detail) in enumerate(reversed(layers)):
        y = idx * 2 + 1
        rect = mpatches.FancyBboxPatch(
            (1.5, y - 0.6), 7, 1.2,
            boxstyle="round,pad=0.1",
            linewidth=1.5,
            edgecolor="#2c3e50",
            facecolor="#ecf0f1",
        )
        ax.add_patch(rect)
        ax.text(5, y + 0.15, name, ha="center", va="center", fontsize=11, fontweight="bold")
        ax.text(5, y - 0.2, detail, ha="center", va="center", fontsize=9)
        if idx < len(layers) - 1:
            ax.annotate("", xy=(5, y + 0.65), xytext=(5, y + 1.35),
                        arrowprops=dict(arrowstyle="->", lw=1.5, color="#2c3e50"))


lstm_layers = [
    ("Input", "input_ids + attention_mask"),
    ("Embedding", f"{lstm_model.vocab_size} x {lstm_model.embedding_dim}"),
    ("BiLSTM", f"{lstm_model.num_layers} sloja, hidden={lstm_model.hidden_dim}"),
    ("Dropout", f"p={lstm_model.dropout_rate}"),
    ("Linear Head", f"{lstm_model.hidden_dim * 2} -> {config.NUM_CLASSES}"),
    ("Softmax", f"{config.NUM_CLASSES} klasa"),
]

bert_layers = [
    ("Input", "input_ids + attention_mask"),
    ("BERT Encoder", config.MODEL_NAME),
    ("Pooler", "CLS reprezentacija (768)"),
    ("Dropout", f"p={config.BERT_DROPOUT}"),
    ("Linear Head", f"768 -> {config.NUM_CLASSES}"),
    ("Softmax", f"{config.NUM_CLASSES} klasa"),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
draw_architecture_diagram(lstm_layers, "LSTM Baseline", axes[0])
draw_architecture_diagram(bert_layers, "BERT Fine-tuning", axes[1])
plt.tight_layout()
plt.show()

## Zaključak

- **LSTM** je lakši baseline model sa znatno manje parametara, pogodan za brze eksperimente
- **BERT** koristi pretrained reprezentacije i ima daleko više parametara, ali obično daje bolju tačnost
- Oba modela vraćaju `logits`, `probs` (softmax) i opcioni `loss` za kompatibilnost sa trening pipeline-om

Sledeći korak: treniranje i evaluacija oba modela na `train/val/test` splitovima.